In [3]:
### IMPORTS
import pandas as pd
import numpy as np
from backtester.pnl import pnl
from backtester.metrics_old import calc_metrics, return_metrics, infer_ann_factor, _max_drawdown, _equity_curve
from utils.io import load_partitioned_parquet
from utils.paths import make_data_path
from backtester.metrics.performance_report import PerformanceReport

In [4]:
###CONFIG PLEASE
INITAL_CAPITAL = 100000
LEVERAGE = 1.0
DELAY_BARS = 1 # 1-bar execution delay
MAKER_FEE = 0.000200
TAKER_FEE = 0.000550
SYMBOL = 'BTCUSDT'
INTERVAL = 60
START = '01/01/2026'
END = None

In [5]:
### THis needs to be a function too
### LOAD DATA

cols = ['mark_close'] # Whatever I want for the strategy 
path_ohlcv = make_data_path('ohlcv', SYMBOL, INTERVAL)
df_ohlcv = load_partitioned_parquet(path_ohlcv, start=START, end=END, columns=cols)

if df_ohlcv.empty:
    raise ValueError("df_ohlcv loaded empty. Check path or date filters.")

path_funding = make_data_path('funding', SYMBOL)
df_funding = load_partitioned_parquet(path_funding,start=START, end=END)

if df_funding.empty:
    raise ValueError("df_funding loaded empty. Check path or date filters.")

df_merge = df_ohlcv.merge(df_funding, how='left', on='timestamp')
df_merge['fundingRate'] = df_merge['fundingRate'].fillna(0)
df_merge = df_merge.sort_index()

mark_close = df_merge["mark_close"].astype("float64").to_numpy()
funding  = df_merge["fundingRate"].astype("float64").to_numpy()


In [6]:
pos = np.ones(len(mark_close)) # always long lol
print(len(pos))

1710


In [7]:
pos = np.zeros_like(mark_close)
for i in range(len(mark_close)):
    if i % 2 == 0:
        pos[i] = 1

In [8]:
pos = np.random.uniform(-1, 1, size=(len(mark_close)))

In [9]:
pnl_df = pnl(df_merge, pos, capital=INITAL_CAPITAL,leverage=LEVERAGE, delay_bars=DELAY_BARS)
print(pnl_df.head(10))
print(pnl_df.tail(5))

                           asset_change (%)  signal (% of equity)  \
timestamp                                                           
2026-01-01 00:00:00+00:00          0.000000              0.150850   
2026-01-01 01:00:00+00:00          0.001777             -0.281512   
2026-01-01 02:00:00+00:00         -0.000459              0.844783   
2026-01-01 03:00:00+00:00         -0.000638              0.768131   
2026-01-01 04:00:00+00:00         -0.003008             -0.596640   
2026-01-01 05:00:00+00:00          0.001283              0.863459   
2026-01-01 06:00:00+00:00         -0.000590             -0.653357   
2026-01-01 07:00:00+00:00         -0.000341             -0.886851   
2026-01-01 08:00:00+00:00          0.001633             -0.261801   
2026-01-01 09:00:00+00:00          0.001227             -0.010429   

                           held_pos (% of equity)  trade (% of equity)  \
timestamp                                                                
2026-01-01 00:00:00+00:

In [10]:
report=PerformanceReport(pnl_df)
print(report.returns.net_return)
print(report.cost.gross_return)
print(report.cost.total_fee_return)
print(report.cost.total_cost_pct_of_net)
print(report.cost.total_cost_return)
print(report.cost.total_funding_return)
print(report.cost.total_fee_pct_of_net)
print(report.cost.cost_to_gross_ratio)
print(report.cost.pct_bars_paying_funding)
print(report.cost.avg_fee_per_bar)
print(report.cost.annualized_turnover)
print(report.cost.fee_drag_on_sharpe)
print(report.cost.funding_drag_on_sharpe)



-0.43658919391639855
0.05512666141521601
0.6206259952705628
1.093812640232118
0.6185738844011646
-2.1543897443453814e-05
1.0974413495338022
11.220956766128891
0.06198830409356725
0.000362939178520797
9884879.48830933
0.11776714910403105
-1.5643815745869372e-06
